In [ ]:
from argparse import Namespace
args = Namespace(
    sim_ckpt="workdir/TPS_Transition1x-rcut12-path_linearOT/PotentialEnergy/epoch=198-step=1894990.ckpt",
    data_dir="data/Transition1x/",
    suffix="",
    num_rollouts=1,
    # out_dir="./test/Transition1x-rcut12-path_linearOT/",
    out_dir="./test/Transition1x-rcut12-path_linearOT/TPS/",
    num_frames=1,
    random_starting_point=True,
    localmask=False,
    tps_condition=True,
    sim_condition=False
    )
device = "cuda"

In [ ]:
import os, torch, tqdm, time
import numpy as np
from mdgen.equivariant_wrapper import EquivariantMDGenWrapper

In [ ]:
os.makedirs(args.out_dir, exist_ok=True)

In [ ]:

from mdgen.dataset import EquivariantTransformerDataset_Transition1x
dataset = EquivariantTransformerDataset_Transition1x(data_dirname=args.data_dir, sim_condition=args.sim_condition, tps_condition=args.tps_condition, stage="train")


In [ ]:
print(dataset[0])

In [ ]:
ckpt = torch.load(args.sim_ckpt, weights_only=False)
hparams = ckpt["hyper_parameters"]
hparams['args'].guided = True
model = EquivariantMDGenWrapper(**hparams)
print(model.model)
model.load_state_dict(ckpt["state_dict"], strict=False)
model.eval().to(device)

In [ ]:
print(ckpt["hyper_parameters"])
print(len(dataset))

In [ ]:
embed_dim = ckpt["hyper_parameters"]['args'].embed_dim

In [ ]:
print(ckpt["hyper_parameters"]['args'].num_heads)

In [ ]:
batch_size = 1
val_loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=batch_size,
    num_workers=0,
    shuffle=True,
)
sample_batch = next(iter(val_loader))


In [ ]:
print(sample_batch.keys())

In [ ]:
print(dataset[499]["x"].shape)

In [ ]:
raise RuntimeError

## Test generative model

In [ ]:
for key in ['species', 'x', 'cell', 'num_atoms', 'mask', 'v_mask', 'species_next', 'x_next', "TKS_mask", "TKS_v_mask"]:
    try:
        sample_batch[key] = sample_batch[key].to(device)
    except:
        print(f"{key} not found")

pred_pos = model.inference(sample_batch)
'''
model.stage = "inference"
prep = model.prep_batch(sample_batch)
B,T,L,_ = prep["latents"].shape
t = torch.ones((B,), device=prep["latents"].device)
print(model.potential_model(prep['latents'], t, **prep['model_kwargs']).sum(dim=2).squeeze(-1)[:,1])
'''

In [ ]:
@torch.no_grad()
def rollout(model, batch):
    expanded_batch = batch
    
    positions, _ = model.inference(expanded_batch)

    # mask_act_space = (batch["mask"] != 0)
    # positions = positions*mask_act_space
    new_batch = {**batch}
    new_batch['x'] = positions
    return positions, new_batch

'''
map_to_chemical_symbol = {
    0: "Cr",
    1: 'Co',
    2: "Ni"
}
'''
map_to_chemical_symbol = {
    0: "H",
    1: 'C',
    2: "N",
    3: "O"

}

In [ ]:
from ase import Atoms
from ase.geometry.geometry import get_distances

all_rollout_atoms_ref_0 = []
all_rollout_atoms = []
all_rollout_atoms_ref = []
start = time.time()
for i_rollout in range(50):
# for i_rollout in range(1):
    idx = np.random.randint(0, len(dataset), 1)[0]
    # idx = 0
    item = dataset.__getitem__(idx)
    batch = next(iter(torch.utils.data.DataLoader([item])))

    for key in ['species', 'x', 'cell', 'num_atoms', 'mask', 'v_mask', 'species_next', 'x_next', "TKS_mask", "TKS_v_mask"]:
        try:
            batch[key] = batch[key].to(device)
        except:
            print(f"{key} not found")

    labels = torch.argmax(batch["species"], dim=3).squeeze(0)
    symbols = [[map_to_chemical_symbol[int(i_elem.to('cpu'))] for i_elem in labels[i_conf]] for i_conf in range(len(labels))]

    pred_pos, _ = rollout(model, batch)
    # print("idx = ", idx, "rollout", i_rollout, pred_pos.shape)

    all_atoms = []
    all_atoms_ref = []
    all_atoms_ref_0 = []
    for t in range(len(pred_pos[0])):
        print("idx = ", idx, "rollout", i_rollout, "t", t)
        formula = "".join(symbols[t])

        # print("t=",t)
        # for i in range(pred_pos.shape[2]):
        #     err = get_distances(batch["x_next"][0][t][i].cpu().numpy(), (pred_pos[0][t].cpu().numpy()[i]), cell=batch['cell'][0][0].cpu().numpy(), pbc=True)[1][0][0]
        #     if err>0.1:
        #         print(pred_pos[0][t].cpu().numpy()[i], batch["x_next"][0][t][i].cpu().numpy(), err, err>0.1)
        atoms = Atoms(formula, positions=pred_pos[0][t].cpu().numpy(), cell=batch['cell'][0][0].cpu().numpy(), pbc=[1,1,1])
        # atoms.set_chemical_symbols(symbols[t])
        all_atoms.append(atoms)
        if args.sim_condition:
            atoms_ref_0 = Atoms(formula, positions=batch["x"][0][t].cpu().numpy(), cell=batch['cell'][0][0].cpu().numpy(), pbc=[1,1,1])
            atoms_ref = Atoms(formula, positions=batch["x_next"][0][t].cpu().numpy(), cell=batch['cell'][0][0].cpu().numpy(), pbc=[1,1,1])
        else:
            atoms_ref = Atoms(formula, positions=batch["x"][0][t].cpu().numpy(), cell=batch['cell'][0][0].cpu().numpy(), pbc=[1,1,1])
        all_atoms_ref.append(atoms_ref)
        if args.sim_condition:
            all_atoms_ref_0.append(atoms_ref_0)
        if args.tps_condition:
            if t == 1:
                err = pred_pos[0][t]-batch["x"][0][t]
                print(torch.abs(err).max(), torch.abs(err).min(), torch.abs(err).mean(), )
                assert not torch.allclose(pred_pos[0][t], batch["x"][0][t])
                assert not np.allclose(pred_pos[0][t].cpu().numpy(), batch["x"][0][t].cpu().numpy())
            else:
                assert torch.allclose(pred_pos[0][t], batch["x"][0][t])
    all_rollout_atoms.append(all_atoms)
    all_rollout_atoms_ref.append(all_atoms_ref)
    if args.sim_condition:
        all_rollout_atoms_ref_0.append(all_atoms_ref_0)


In [ ]:

'''
for i_rollout in range(10):
    print("rollout", i_rollout, pred_pos.shape)
    all_atoms = all_rollout_atoms[i_rollout]
    all_atoms_ref = all_rollout_atoms_ref[i_rollout]
    for t in range(len(pred_pos[0])):
        print("t=",t)
        atoms = all_atoms[t]
        atoms_ref = all_atoms_ref[t]
        for i in range(atoms.positions.shape[0]):
            err = get_distances(atoms_ref.positions[i], atoms.positions[i], cell=atoms.cell, pbc=True)[1][0][0]

            if err>0.1:
                print(atoms.positions[i], atoms_ref.positions[i], err, err>0.1)
        
'''

In [ ]:
import shutil, os
from ase.io import write
all_atom_filename = os.path.join(args.out_dir, "all_atoms.xyz")
ref_all_atom_filename = os.path.join(args.out_dir, "ref_all_atoms.xyz")
if os.path.exists(all_atom_filename): os.remove(all_atom_filename)
if os.path.exists(ref_all_atom_filename): os.remove(ref_all_atom_filename)
for i in range(50):
    dirname = os.path.join(args.out_dir, f"rollout_{i}")
    if not os.path.exists(dirname):
        os.makedirs(dirname)
    filename_0 = os.path.join(dirname, "gentraj_0.xyz")
    filename = os.path.join(dirname, "gentraj_1.xyz")
    filename_ref_0 = os.path.join(dirname, "reftraj_0.xyz")
    filename_ref = os.path.join(dirname, "reftraj_1.xyz")
    if os.path.exists(filename):
    #     shutil.move(filename_0, os.path.join(dirname, "bck.0.gentraj_0.xyz"))
        os.remove(filename)
    #     shutil.move(filename_ref_0, os.path.join(dirname, "bck.0.reftraj_0.xyz"))
        os.remove(filename_ref)
    print(len(all_rollout_atoms[i]))
    assert not np.allclose(all_rollout_atoms[i][1].positions, all_rollout_atoms_ref[i][1].positions)
    for atoms in all_rollout_atoms[i]:
        atoms.set_cell(np.eye(3,3)*10)
        write(filename, atoms, append=True)
        write(all_atom_filename, atoms, append=True)
    for ref_atoms in all_rollout_atoms_ref[i]:
        print(i, filename_ref)
        ref_atoms.set_cell(np.eye(3,3)*10)
        write(filename_ref, ref_atoms, append=True)
        write(ref_all_atom_filename, ref_atoms, append=True)
    '''
    if args.sim_condition:
        for ref_atoms_0 in all_rollout_atoms_ref_0[i]:
            print(i, filename_ref_0)
            write(filename_0, ref_atoms_0)
            write(filename_ref_0, ref_atoms_0)
    '''

In [ ]:
raise RuntimeError

### Generate trajectory

In [ ]:
from copy import deepcopy
@torch.no_grad()
def rollout(model, batch):
    expanded_batch = batch
    
    positions, _ = model.inference(expanded_batch)

    # mask_act_space = (batch["mask"] != 0)
    # positions = positions*mask_act_space
    new_batch = {}
    for k in batch.keys():
        new_batch[k] = deepcopy(batch[k])
    new_batch['x'] = positions
    return positions, new_batch


map_to_chemical_symbol = {
    0: "Cr",
    1: 'Co',
    2: "Ni"
}

In [ ]:
from mace.calculators import MACECalculator
calculator = MACECalculator(
    model_path="./MACE-matpes-r2scan-omat-ft.model",
    device="cuda",
    default_dtype="float32",
)

In [ ]:
idx = np.random.randint(0, len(dataset), 1)[0]
print(idx)

In [ ]:

def get_action_space(atoms, lattice_parameter: float = 3.528):
    a = lattice_parameter
    actions = []
    dist_mat = atoms.get_all_distances(mic=True)

    crit = np.sum(dist_mat < a/np.sqrt(2)*1.2, axis=1)
    vacancy_l = np.argwhere(crit != 13).T[0]

    def test(i, vec):
        test = atoms.copy()
        pos_test = test.get_positions()
        pos_test[i] += vec
        test.set_positions(pos_test)

        return np.sum(test.get_distances(i, range(len(test)), mic=True) < 0.8) == 1

    acts = np.array([[1, 1, 0],
                     [1, -1, 0],
                     [-1, 1, 0],
                     [-1, -1, 0],
                     [1, 0, 1],
                     [1, 0, -1],
                     [-1, 0, 1],
                     [-1, 0, -1],
                     [0, 1, 1],
                     [0, 1, -1],
                     [0, -1, 1],
                     [0, -1, -1]])*a/2*0.8

    for i in vacancy_l:
        for vec in acts:
            vacant = test(i, vec)
            if vacant:
                actions.append([i]+vec.tolist())

    return actions


In [ ]:
17.64/3.528

In [ ]:
from ase.build import bulk
from ase.visualize import view

# 1) Create a single-unit-cell FCC crystal of copper (Cu)
#    a = lattice constant in Å
fcc = bulk('Cu',        # element symbol
           'fcc',       # crystal structure
           a=3.528,       # lattice constant
           cubic=True)  # enforce a cubic cell

# 2) (Optional) Make a supercell
fcc_super = fcc * (5, 5, 5)
fcc_super.positions -= np.ones(3)*0.5 @ fcc_super.cell
# 3) Write to disk
from ase.io import write
write('fcc_cu_5x5x5.xyz',    fcc_super)

# 4) (Optional) Visualize in ASE GUI
view(fcc_super)


In [ ]:
def pseudo_relax(atoms, atoms_ref):
    for i_atom in range(len(atoms.positions)):
        all_dist = get_distances(atoms.positions[i_atom], atoms_ref.positions, cell=atoms.cell, pbc=True)[1]
        j_ref = np.argmin(all_dist)
        assert np.min(all_dist)<1
        atoms.positions[i_atom] = atoms_ref.positions[j_ref]
    return atoms

In [ ]:
from ase import Atoms
from ase.geometry.geometry import get_distances
from ase.optimize import BFGS, FIRE       # two popular optimizers

all_rollout_atoms = []
all_rollout_atoms_relax = []
start = time.time()

item = dataset.__getitem__(idx, start_i_traj=0)
batch = next(iter(torch.utils.data.DataLoader([item])))
for key in ['species', 'x', 'cell', 'num_atoms', 'mask', 'v_mask', 'species_next', 'x_next', "TKS_mask", "TKS_v_mask"]:
    try:
        batch[key] = batch[key].to(device)
    except:
        print(f"{key} not found")
print(torch.from_numpy(np.loadtxt(dataset.traj_act_space[idx])).to(torch.long)[0])
log_filename = os.path.join(args.out_dir, "fire.log")
if os.path.exists(log_filename): os.remove(log_filename)
for i_rollout in range(18):
# for i_rollout in range(1):

    labels = torch.argmax(batch["species"], dim=3).squeeze(0)
    symbols = [[map_to_chemical_symbol[int(i_elem.to('cpu'))] for i_elem in labels[i_conf]] for i_conf in range(len(labels))]
    pred_pos, new_batch = rollout(model, batch)

    assert torch.allclose(pred_pos, new_batch['x'])
    assert len(pred_pos[0]) == 1
    if i_rollout == 0:
        for t in range(1):
            formula = "".join(symbols[t])
            atoms_ref = Atoms(formula, positions=batch["x"][0][t].cpu().numpy(), cell=batch['cell'][0][0].cpu().numpy(), pbc=[1,1,1])
            actions = torch.from_numpy(np.array(get_action_space(atoms_ref, ))[:,0]).unsqueeze(0).to(torch.long)
            print(actions)
        all_rollout_atoms.append(atoms_ref)
        all_rollout_atoms_relax.append(atoms_ref)

    relaxed_pos = []
    for t in range(1):
        formula = "".join(symbols[t])
        atoms = Atoms(formula, positions=pred_pos[0][t].cpu().numpy(), cell=batch['cell'][0][0].cpu().numpy(), pbc=[1,1,1])
        # atoms.set_chemical_symbols(symbols[t])
        all_rollout_atoms.append(atoms)
        # atoms.calc = calculator
        # opt = FIRE(atoms, logfile=log_filename)
        # opt.run(fmax=0.005)
        atoms = pseudo_relax(atoms, fcc_super)
        all_rollout_atoms_relax.append(atoms)
        relaxed_pos.append(atoms.positions)

    print("idx = ", idx, "rollout", i_rollout, len(all_rollout_atoms[i_rollout+1]))

    new_batch['x'] = torch.from_numpy(np.array(relaxed_pos)).unsqueeze(0).to(torch.float32)
    print(get_distances(batch['x'][0,0].cpu().numpy(), new_batch['x'][0,0].cpu().numpy(), cell=batch['cell'][0,0].cpu().numpy(), pbc=True)[1].diagonal().max(), np.where(get_distances(batch['x'][0,0].cpu().numpy(), new_batch['x'][0,0].cpu().numpy(), cell=batch['cell'][0,0].cpu().numpy(), pbc=True)[1].diagonal() > 1.5))
    # assert not torch.allclose(new_batch['x'].to(device), batch['x'], atol=1)

    actions = torch.from_numpy(np.array(get_action_space(atoms, ))[:,0]).unsqueeze(0).to(torch.long)
    # print(actions)
    mask_dict = dataset.mask_from_actions(idx, actions, start_i_traj=0)
    for k in mask_dict.keys():
        new_batch[k] = deepcopy(mask_dict[k])
    ## compare the new batch against the next step of the reference trajectory
    if i_rollout < 20:
        item_ref = dataset.__getitem__(idx, start_i_traj=i_rollout+1)
        batch_ref = next(iter(torch.utils.data.DataLoader([item_ref])))
        for k in ['species', 'x', 'cell', 'num_atoms', 'mask', 'v_mask', 'species_next', "TKS_mask", "TKS_v_mask"]:
            batch_ref[k] = batch_ref[k].to(device)
            new_batch[k] = new_batch[k].to(device)
            if "mask" in k:
                assert torch.allclose(batch_ref[k]!=0 , new_batch[k]!=0), f"{k}"
            elif "x" == k:
                dist_mat = get_distances(batch_ref[k][0,0].cpu().numpy(), new_batch[k][0,0].cpu().numpy(), cell=batch_ref['cell'][0,0].cpu().numpy(), pbc=True)[1].diagonal()
                if not np.all(dist_mat < 1):
                    for i_atom in range(len(batch_ref[k][0,0])):
                        print(i_atom, batch_ref[k][0,0,i_atom].cpu().numpy(), new_batch[k][0,0,i_atom].cpu().numpy(), dist_mat[i_atom], dist_mat[i_atom] >= 1)
                    raise Exception("x not right")
                new_batch['x'] = deepcopy(batch_ref['x'])
            else:
                assert torch.allclose(batch_ref[k], new_batch[k]), f"{k}"       
        ## update batch with new batch
        # for k in new_batch.keys():
        #     batch[k] = new_batch[k]
        batch_ref['x_next'] = batch_ref['x_next'].to(device)
        batch = batch_ref


In [ ]:

print(torch.where(batch['mask'] != 0))

In [ ]:
import shutil
from ase.io import write
from ase import Atoms
dataset.num_frames = 20
print(idx)
item = dataset.__getitem__(idx, start_i_traj=0)
batch = next(iter(torch.utils.data.DataLoader([item])))


labels = torch.argmax(batch["species"], dim=3).squeeze(0)
symbols = [[map_to_chemical_symbol[int(i_elem.to('cpu'))] for i_elem in labels[i_conf]] for i_conf in range(len(labels))]
all_atoms_ref = []
for t in range(len(batch['x'][0])):
    formula = "".join(symbols[t])
    atoms_ref = Atoms(formula, positions=batch["x"][0][t].cpu().numpy(), cell=batch['cell'][0][0].cpu().numpy(), pbc=[1,1,1])
    all_atoms_ref.append(atoms_ref)

filename_ref = os.path.join(args.out_dir, "reftraj_.xyz")
write(filename_ref, all_atoms_ref)
dataset.num_frames = 1

In [ ]:
print(len(batch['x'][0]))

In [ ]:
'''
from ase.io import read
u1 = idx // 100
k = idx % 100
traj_filename = os.path.join("data/CrCoNi_sims/", f"testing-{u1}-{k}.extxyz")
atoms_list = read(traj_filename, index=":")
write(os.path.join(args.out_dir, "src_reftraj_.xyz"), atoms_list)
'''

In [ ]:

filename = os.path.join(args.out_dir, "gentraj_.xyz")
if os.path.exists(filename): os.remove(filename)
#     shutil.move(filename, os.path.join(args.out_dir, "bck.0.gentraj.xyz"))
#     shutil.move(filename_ref, os.path.join(args.out_dir, "bck.0.reftraj.xyz"))
write(filename, all_rollout_atoms)

filename = os.path.join(args.out_dir, "gentraj_relax.xyz")
if os.path.exists(filename): os.remove(filename)
write(filename, all_rollout_atoms_relax)